# Reliable score — rule làm nhãn

Đầu vào `df` dạng dài, mỗi dòng một (shop, snapshot): `cus_id`, `date` (snapshot, **liên tục** từng tháng), `tong_don_mean_l3m` (TB đơn/tháng trong 3 tháng gần nhất tại snapshot), `tenure` (thâm niên, tháng). Shop không có dòng ở snapshot nào thì coi = 0 đơn.

- base = `tong_don_mean_l3m` tại snapshot *t*; fut = `tong_don_mean_l3m` tại snapshot *t*+4.
- Prediction set cohort *t*: base ≥ 5, thâm niên ≥ 3.
- bad = 1 nếu fut = 0, hoặc (fut / base) / m < 0,3; m tính trên S0 (base ≥ 10, thâm niên ≥ 6).
- Chỉ lấy cohort đã có snapshot *t*+4.

In [1]:
import pandas as pd


def make_labels(df, fut_lag=4, min_orders=5, min_tenure=3, s0_orders=10, s0_tenure=6, threshold=0.3):
    o = df.pivot_table(index="cus_id", columns="date", values="tong_don_mean_l3m", aggfunc="first").fillna(0)
    ten = df.pivot_table(index="cus_id", columns="date", values="tenure", aggfunc="first")
    months = sorted(o.columns)
    out = []
    for i in range(len(months) - fut_lag):                  # cohort đã có snapshot t+fut_lag
        t, f = months[i], months[i + fut_lag]
        x = pd.DataFrame({"cus_id": o.index, "date": t, "base_mean": o[t].values, "fut_mean": o[f].values,
                          "tenure": ten[t].reindex(o.index).values})
        x = x[(x["base_mean"] >= min_orders) & (x["tenure"] >= min_tenure)]
        s0 = (x["base_mean"] >= s0_orders) & (x["tenure"] >= s0_tenure)
        m = x.loc[s0, "fut_mean"].sum() / x.loc[s0, "base_mean"].sum()   # hệ số thị trường
        x["ratio"] = x["fut_mean"] / x["base_mean"] / m
        x["label"] = ((x["fut_mean"] == 0) | (x["ratio"] < threshold)).astype(int)
        out.append(x)
    return pd.concat(out, ignore_index=True)

In [ ]:
def make_label_V2(df, fut_lag=4, min_orders=5, min_tenure=3, s0_orders=10, s0_tenure=6, threshold=0.3):
    """Như make_labels bản gốc (cột orders = đơn tháng), chỉ đổi ratio = orders(T+fut_lag) / orders(T) / m."""
    o = df.pivot_table(index="cus_id", columns="date", values="orders", aggfunc="sum").fillna(0)
    ten = df.pivot_table(index="cus_id", columns="date", values="tenure", aggfunc="first")
    months = sorted(o.columns)
    out = []
    for i in range(2, len(months) - fut_lag):               # cohort đủ 3 tháng baseline + có snapshot T+fut_lag
        t, f, base = months[i], months[i + fut_lag], months[i - 2:i + 1]
        x = o[(o[base] > 0).all(axis=1)]
        x = pd.DataFrame({"cus_id": x.index, "date": t, "base_mean": x[base].mean(axis=1).values,
                          "order_t": x[t].values, "order_fut": x[f].values,
                          "tenure": ten[t].reindex(x.index).values})
        x = x[(x["base_mean"] >= min_orders) & (x["tenure"] >= min_tenure)]
        s0 = (x["base_mean"] >= s0_orders) & (x["tenure"] >= s0_tenure)
        m = x.loc[s0, "order_fut"].sum() / x.loc[s0, "order_t"].sum()    # hệ số thị trường
        x["ratio"] = x["order_fut"] / x["order_t"] / m
        x["label"] = ((x["order_fut"] == 0) | (x["ratio"] < threshold)).astype(int)
        out.append(x)
    return pd.concat(out, ignore_index=True)

Ví dụ giả lập: 6 tháng → chỉ cohort tháng 3 đủ nhãn.

In [2]:
months = [20260101, 20260201, 20260301, 20260401, 20260501, 20260601]
shops = {"A": [20, 20, 20, 20, 20, 20],     # ổn định -> good
         "E": [30, 30, 30, 30, 30, 30],     # ổn định -> good
         "B": [20, 20, 20, 3, 2, 1],        # giảm mạnh -> bad
         "C": [20, 20, 20, 0, 0, 0],        # ngừng giao -> bad
         "D": [3, 3, 3, 3, 3, 3]}           # < 5 đơn/tháng -> ngoài prediction set
df = pd.DataFrame([{"cus_id": c, "date": d, "orders": n, "tenure": 12}
                   for c, v in shops.items() for d, n in zip(months, v)])
# make_labels(df)

In [3]:
df

,cus_id,date,orders,tenure
0,A,20260101,20,12
1,A,20260201,20,12
2,A,20260301,20,12
3,A,20260401,20,12
4,A,20260501,20,12
5,A,20260601,20,12
6,E,20260101,30,12
7,E,20260201,30,12
8,E,20260301,30,12
9,E,20260401,30,12
